In [1]:
import pandas as pd
import numpy as np
import functions as f

pd.set_option("display.width", 2000)
pd.set_option("display.max_columns", 200)

# Park1 – Ignitis Renewables  
## Shunt Reactor Operational Analysis

### Task Goal

The goal of this task is to analyze real operational data from a high-voltage shunt reactor, evaluate data quality, detect anomalies, and clearly explain the findings from an electrical and operational perspective.

---

### Objectives

- Perform data quality checks (missing values, gaps, corrupted data).
- Detect outliers and abnormal operating patterns.
- Evaluate reactor behavior over the analyzed time period.
- Identify periods of unusual or abnormal operation.
- Compare measurements against design limits:
  - Voltage: **30 kV ±10%**
  - Current: **125.1 A ±10%**
- Create clear visualizations to communicate key insights.

---


In [2]:
df = pd.read_csv("data/Park1_SR_data.csv", sep=";", dtype=str)

df = df.replace("\t", "", regex=True)
df["TimeStamp"] = pd.to_datetime(
    df["TimeStamp"],
    format="mixed",
    errors="coerce"
)
for col in df.columns:
    if col != "TimeStamp":
        df[col] = df[col].str.replace(",", ".", regex=False).astype(float)

## Data Overview

This dataset contains time-series operational measurements from a high-voltage shunt reactor.  
The data represents real grid operation over a defined time period.

---

## Variables Description

### Voltage (kV)

- **SR_U12_kV, SR_U23_kV, SR_U31_kV** – Line-to-line voltages.  
- **SR_U1_kV, SR_U2_kV, SR_U3_kV** – Phase voltages.

Used to verify voltage stability and check compliance with the 30 kV ±10% design specification Line-to-line voltages.

---

### Current (A)

- **SR_I1_A, SR_I2_A, SR_I3_A** – Phase currents flowing through the reactor.

Used to evaluate loading condition and compliance with 125.1 A ±10% rating.

---

### Reactive Power (MVAr)

- **SR_avg_Q_MVar**
- **SR_max_Q_MVar**

Indicates the amount of reactive power absorbed by the reactor.

---

### Derived Metrics

- **Voltage imbalance (%)**
- **Current imbalance (%)**

Used to assess phase symmetry and detect abnormal electrical conditions.

In [3]:
print(df.head())

            TimeStamp  SR_avg_Q_MVar  SR_max_Q_MVar  SR_I1_A  SR_I2_A  SR_I3_A  SR_U12_kV  SR_U23_kV  SR_U31_kV  SR_U1_kV  SR_U2_kV  SR_U3_kV
0 2025-01-21 14:00:00          6.617          6.644  130.097  129.475  128.854     29.434     29.469     29.477    17.006    16.991    17.020
1 2025-01-21 14:10:00          6.639          6.647  130.127  129.525  128.904     29.502     29.533     29.548    17.047    17.032    17.058
2 2025-01-21 14:20:00          6.622          6.646  130.252  129.734  129.113     29.470     29.495     29.506    17.025    17.013    17.039
3 2025-01-21 14:30:00          6.611          6.630  130.252  129.734  129.113     29.427     29.448     29.463    17.008    16.987    17.013
4 2025-01-21 14:40:00          6.619          6.625  130.203  129.636  128.990     29.455     29.479     29.487    17.016    17.003    17.029


In [4]:
df_check = f.df_check(df)


DATAFRAME BASIC INFO
Shape: (47598, 12)

Dtypes:
TimeStamp        datetime64[ns]
SR_avg_Q_MVar           float64
SR_max_Q_MVar           float64
SR_I1_A                 float64
SR_I2_A                 float64
SR_I3_A                 float64
SR_U12_kV               float64
SR_U23_kV               float64
SR_U31_kV               float64
SR_U1_kV                float64
SR_U2_kV                float64
SR_U3_kV                float64
dtype: object

TIMESTAMP CHECK
Corrupted timestamps: 0
Duplicate timestamps: 0
Detected sampling interval: 0 days 00:10:00
Timestamp gaps: 59

DUPLICATE ROWS
Fully duplicated rows: 0

MISSING VALUES
Total missing values: 11

Missing per column:
SR_avg_Q_MVar    1
SR_max_Q_MVar    1
SR_I1_A          1
SR_I2_A          1
SR_I3_A          1
SR_U12_kV        1
SR_U23_kV        1
SR_U31_kV        1
SR_U1_kV         1
SR_U2_kV         1
SR_U3_kV         1
dtype: int64

ZERO VALUES
Total zero values: 2862

Zero values per column:
SR_avg_Q_MVar    264
SR_max_Q_MVar   

## Reactor Behavior – Key Insights

### Data Availability

- 59 timestamp gaps were detected.
- Most gaps are short (10–40 minutes), but several long interruptions occurred (up to ~4 days).
- These long gaps indicate communication or data acquisition issues rather than operational instability.

---

### Data Loss Events

- 7 full data-loss blocks were detected.
- Total duration of missing data: ~1 day 17 hours.
- These periods contain 0 values in messurments, assuming it as data loss.

---

### Reactor Operating States

- 5 short Reactor OFF events were identified (total duration ~20 minutes).
- These appear as controlled switching events.
- Reactor was operational and stable for the majority of the analyzed period.

---

### Design Limit Compliance

**Voltage (27–33 kV):**
- 1983 short-duration violations detected.
- No sustained overvoltage condition observed.

**Current (112.6–137.6 A):**
- 1995 short violation blocks detected.
- Most occurred during switching transitions.

---

### Extreme Spike Event

An abnormal event was detected where:

- Phase currents exceeded ~ **1300 A**
- Line and phase voltages simultaneously reached extremely high values (over 300 kV)

These values are physically unrealistic for a 30 kV shunt reactor system and indicate:
- Measurement corruption,
- Scaling error,
- Or telemetry malfunction.

The spikes (I ≈ 1300 A, U ≈ 300 kV, Q up to ~70 MVAr) are considered non-physical because they exceed the 30 kV / 125 A design ratings by ~10× and break basic electrical consistency: if U and I increased by ~10×, reactive power would scale to ~100× (~675 MVAr), but the recorded Q does not. The events are short and the signals return immediately to normal without a protection/shutdown signature, which is typical of scaling or telemetry errors rather than real reactor behavior.

---

### Overall Assessment

The reactor operates in a generally stable manner within its design limits.  
Most detected anomalies are short-duration switching events.  

The only severe abnormal readings (1300 A and >300 kV) are clearly non-physical and most likely caused by measurement or communication errors rather than true electrical stress on the equipment.

In [5]:
f.df_event_report(df)


TIMESTAMP GAPS
Gaps detected: 59
Rows 37 → 38 | 2025-01-21 20:10:00 → 2025-01-21 21:00:00 | Missing: 0 days 00:40:00
Rows 246 → 247 | 2025-01-23 07:40:00 → 2025-01-23 08:00:00 | Missing: 0 days 00:10:00
Rows 273 → 274 | 2025-01-23 12:20:00 → 2025-01-23 13:00:00 | Missing: 0 days 00:30:00
Rows 408 → 409 | 2025-01-24 11:20:00 → 2025-01-24 11:40:00 | Missing: 0 days 00:10:00
Rows 409 → 410 | 2025-01-24 11:40:00 → 2025-01-24 12:00:00 | Missing: 0 days 00:10:00
Rows 410 → 411 | 2025-01-24 12:00:00 → 2025-01-24 12:20:00 | Missing: 0 days 00:10:00
Rows 425 → 426 | 2025-01-24 14:40:00 → 2025-01-24 15:10:00 | Missing: 0 days 00:20:00
Rows 429 → 430 | 2025-01-24 15:40:00 → 2025-01-24 16:00:00 | Missing: 0 days 00:10:00
Rows 457 → 458 | 2025-01-24 20:30:00 → 2025-01-24 20:50:00 | Missing: 0 days 00:10:00
Rows 826 → 827 | 2025-01-27 10:10:00 → 2025-01-27 10:40:00 | Missing: 0 days 00:20:00
Rows 1121 → 1122 | 2025-01-29 11:40:00 → 2025-01-29 12:00:00 | Missing: 0 days 00:10:00
Rows 1122 → 1123 | 2

In [6]:
df = df.dropna().reset_index(drop=True)
df.shape

(47597, 12)

## Visualisation

All analysis results are presented as interactive visualisations deployed in the cloud.  

The dashboard allows dynamic exploration of:
- Time-series signals (voltage, current, reactive power)
- Reactor OFF events
- Full data loss blocks
- Design limit violations
- Timestamp gaps
- Imbalance metrics

You can zoom, filter, and inspect exact values directly in the browser.

🔗 **Interactive dashboard:**
https://karolisvaivada-shunt-reactor-system-eda-srcapp-rwisyt.streamlit.app/

In [ ]:
cols = df.select_dtypes(include="number").columns

figs = []

for col in cols:
    figs.append(
        f.reactor_full_visualization_interactive(df, col)
    )

for fig in figs:
    fig.show()

In [ ]:
df = f.calculate_imbalance(df)

In [ ]:
fig = f.plot_imbalance(df)
fig.show()

## Conclusion

### Key Findings

- The reactor operates mostly in a stable range around nominal voltage (~30 kV) and current (~125 A).
- Multiple **timestamp gaps (59)** indicate data continuity issues, including several long interruptions.
- **7 full data loss blocks** (all signals = 0) were detected, totaling ~1 day 17 hours.
- **Reactor OFF events** are rare and short (5 blocks, total ~20 minutes).
- Design limit violations are limited in number and mostly short-duration.
- Extreme spikes (up to ~1300 A, ~300 kV, ~70 MVAr) are inconsistent with physical reactor behavior and are likely measurement or scaling errors.
- High number of IQR outliers is mainly driven by spiked values or at the any event. 

### Identified Issues

- Data quality problems (gaps and zero blocks).
- Occasional unrealistic spikes suggesting telemetry, scaling, or historian issues.
- No protection trip signature observed during extreme spikes.

### Assumptions

- Nominal reactor rating: 30 kV ±10%, 125.1 A ±10%.
- Sampling interval assumed constant based on dominant timestamp frequency.
- Extreme values above realistic engineering limits are treated as non-physical.

### Additional Data Recommended

To better assess reactor condition:

- Breaker status (open/closed) signals.
- Protection relay event logs (trip, alarm records).
- CT/VT scaling configuration and raw secondary values.
- Temperature and vibration monitoring of reactor.
- Reactor thermal model data.
- Harmonics and power factor measurements.

These additional signals would allow clearer separation between real electrical events and measurement
